In [ ]:
import pandas as pd
import sys
from pandas import read_csv
import matplotlib.pyplot as plt
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
import numpy as np
from pathlib import Path

In [68]:
import unicodedata


def normalize_text(text):
    text = str(text).strip()
    text = text.replace('Đ', 'D').replace('đ', 'd')
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text.lower()


def rename_columns(df, column_aliases=None):
    # Mac dinh cho cac bo du lieu co cau truc giong VN_index.csv
    default_aliases = {
        'ngay': 'time',
        'lan cuoi': 'close',
        'mo': 'open',
        'cao': 'high',
        'thap': 'low',
        'kl': 'volume',
        '% thay doi': 'pct_change',
    }

    aliases = default_aliases.copy()
    if column_aliases:
        aliases.update({normalize_text(k): v for k, v in column_aliases.items()})

    rename_map = {}
    for col in df.columns:
        normalized_col = normalize_text(col)
        if normalized_col in aliases:
            rename_map[col] = aliases[normalized_col]

    return df.rename(columns=rename_map)


def parse_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    text = str(value).strip().replace(',', '')
    if text == '':
        return np.nan
    return float(text)


def convert_to_datetime(df, time_col='time', date_format='%d/%m/%Y'):
    df = df.copy()
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col], format=date_format, errors='coerce')
    return df


def preprocess_volume(df, volume_col='volume'):
    def convert_volume(volume):
        if pd.isna(volume):
            return np.nan
        if isinstance(volume, (int, float, np.number)):
            return float(volume)

        text = str(volume).strip().replace(',', '')
        if text == '':
            return np.nan

        multiplier = {'K': 1e3, 'M': 1e6, 'B': 1e9}
        suffix = text[-1].upper()
        if suffix in multiplier:
            return float(text[:-1]) * multiplier[suffix]
        return float(text)

    df = df.copy()
    if volume_col in df.columns:
        df[volume_col] = df[volume_col].apply(convert_volume)
    return df


def preprocess_numeric_columns(df, numeric_cols=('close', 'open', 'high', 'low')):
    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_number)
    return df


def preprocess_pct_change(df, pct_col='pct_change'):
    df = df.copy()
    if pct_col in df.columns:
        cleaned = (
            df[pct_col]
            .astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
            .replace({'': np.nan})
        )
        df[pct_col] = cleaned.astype(float)
    return df


def preprocess_market_dataframe(
    df,
    column_aliases=None,
    numeric_cols=('close', 'open', 'high', 'low'),
    time_col='time',
    volume_col='volume',
    pct_col='pct_change',
    date_format='%d/%m/%Y'
    ):
    df = rename_columns(df, column_aliases=column_aliases)
    df = convert_to_datetime(df, time_col=time_col, date_format=date_format)
    df = preprocess_numeric_columns(df, numeric_cols=numeric_cols)
    df = preprocess_volume(df, volume_col=volume_col)
    df = preprocess_pct_change(df, pct_col=pct_col)
    if time_col in df.columns:
        df = df.sort_values(time_col).reset_index(drop=True)
    return df

In [ ]:
vnindex_path = "D:\\UIT\\1003_EPA-Project_UIT\\dataset\\VN_index.csv"
vn30_path = "D:\\UIT\\1003_EPA-Project_UIT\\dataset\\vn30_index.csv"
vnindex_df = read_csv(vnindex_path)
vn30_df = read_csv(vn30_path)

In [ ]:
vnindex_df = preprocess_market_dataframe(vnindex_df)
vn30_df = preprocess_market_dataframe(vn30_df)

In [ ]:
vnindex_df.to_csv("D:\\UIT\\1003_EPA-Project_UIT\\dataset\\vn_index.csv", index=False)
vn30_df.to_csv("D:\\UIT\\1003_EPA-Project_UIT\\dataset\\vn30_index.csv", index=False)


In [46]:
kospi_path = "D:\\UIT\\1003_EPA-Project_UIT\\dataset\\KOSPI_index.csv"

In [ ]:
import yfinance as yf

In [77]:
kospi = yf.Ticker("^KS11")
kospi_df = kospi.history(start="2008-01-01", end="2026-01-01", interval="1d")

In [78]:
kospi_df.to_csv(kospi_path, index=True)

In [71]:
kospi_df = read_csv(kospi_path)

In [81]:
def rename_kospi_columns(df):
    column_map = {
        'Date': 'time',
        'Datetime': 'time',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume',
    }
    return df.rename(columns=column_map)


def add_pct_change_from_close(df, close_col='close', pct_col='pct_change'):
    df = df.copy()
    if close_col in df.columns:
        df[pct_col] = df[close_col].pct_change() * 100
    return df


def preprocess_kospi_to_vn_format(df):
    df = df.copy()

    # Neu thoi gian dang o index (truong hop lay tu yfinance.history)
    if 'Date' not in df.columns and 'Datetime' not in df.columns and 'time' not in df.columns:
        if isinstance(df.index, pd.DatetimeIndex):
            df = df.reset_index()

    df = rename_kospi_columns(df)

    # Neu reset_index tao cot 'index' chua thoi gian
    if 'time' not in df.columns and 'index' in df.columns:
        df = df.rename(columns={'index': 'time'})

    if 'time' in df.columns:
        time_parsed = pd.to_datetime(df['time'], errors='coerce')
        if time_parsed.dt.tz is not None:
            time_parsed = time_parsed.dt.tz_localize(None)
        df['time'] = time_parsed.dt.normalize()

    # Loai bo cot khong co trong schema VN_index
    drop_cols = [col for col in ['Dividends', 'Stock Splits'] if col in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    for col in ['close', 'open', 'high', 'low', 'volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = add_pct_change_from_close(df, close_col='close', pct_col='pct_change')

    ordered_cols = ['time', 'close', 'open', 'high', 'low', 'volume', 'pct_change']
    existing_cols = [col for col in ordered_cols if col in df.columns]
    df = df[existing_cols]

    if 'time' in df.columns:
        df = df.sort_values('time')

    return df.reset_index(drop=True)


if 'kospi_df' not in locals():
    kospi_df = read_csv(kospi_path)

kospi_vn_df = preprocess_kospi_to_vn_format(kospi_df)
kospi_vn_df.head()

,time,close,open,high,low,volume,pct_change
0,2008-01-02,1853.449951,1891.449951,1892.500000,1852.780029,247100,NaN
1,2008-01-03,1852.729980,1834.439941,1858.079956,1821.609985,253700,-0.038845
2,2008-01-04,1863.900024,1853.540039,1869.760010,1824.410034,299100,0.602896
3,2008-01-07,1831.140015,1815.729980,1840.989990,1814.349976,268100,-1.757606
4,2008-01-08,1826.229980,1838.640015,1840.619995,1818.689941,296600,-0.268141


In [83]:
kospi_vn_df.to_csv(kospi_path, index=False)